<a href="https://colab.research.google.com/github/mybright107/workflow_python/blob/main/btaa_SPC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openpyxl pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import csv

INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/btaa_spc/SPC_overlap5.csv"

In [ ]:
# ============================================================
# Overlap Combiner (CSV -> CSV version)
# ============================================================
# 1. Reads an input CSV
# 2. Walks rows pairwise; rows with the same MatchKey as the
#    next row are combined into a single output row
# 3. Writes a plain CSV (no Excel, no colors) with all original
#    columns plus a trailing "Notes" column
# ============================================================

import os

# Extract the directory from INPUT_FILE
OUTPUT_DIR = os.path.dirname(INPUT_FILE)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "SPC_overlap5_output.csv")

MATCH_COL = "MatchKey"
CTRL_COL = "Control Number(001)"
BRANCH_COL = "Branch"


def sniff_delimiter(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        sample = f.read(4096)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",\t").delimiter
    except csv.Error:
        return ","


def pick_row(group, columns):
    """From a group of 2+ rows sharing a MatchKey:
    - If exactly one row's control number ends in '3731', keep only that row.
    - If multiple rows end in '3731', combine those rows' Control Number
      and Branch values into one row (based on the first 3731 row for all
      other columns), and drop every non-3731 row in the group.
    - If none end in '3731', keep the first row and flag it for review.
    In all cases every other row in the group is deleted (no other
    column merging)."""
    notes = []

    ctrls = [str(r[CTRL_COL]).strip() for r in group]
    ends_3731 = [c.endswith("3731") for c in ctrls]
    matches = [r for r, e in zip(group, ends_3731) if e]

    if len(matches) == 1:
        keeper_vals = {col: matches[0][col] for col in columns}
    elif len(matches) > 1:
        base = matches[0]
        keeper_vals = {col: base[col] for col in columns}
        combined_ctrls = [str(r[CTRL_COL]).strip() for r in matches]
        combined_branches = [str(r[BRANCH_COL]).strip() for r in matches if not pd.isna(r[BRANCH_COL])]
        keeper_vals[CTRL_COL] = "; ".join(dict.fromkeys(combined_ctrls))
        keeper_vals[BRANCH_COL] = "; ".join(dict.fromkeys(combined_branches))
        notes.append(f"Combined {len(matches)} rows ending in 3731 (Control Number + Branch merged)")
    else:
        keeper_vals = {col: group[0][col] for col in columns}
        notes.append(
            "REVIEW: no control number in this "
            f"{len(group)}-row match group ends in 3731 "
            f"({', '.join(ctrls)}) - kept first row by default"
        )

    result = {col: ("" if pd.isna(v) else str(v).strip()) for col, v in keeper_vals.items()}
    result["Notes"] = "; ".join(notes)
    return result


def main():
    delim = sniff_delimiter(INPUT_FILE)
    df = pd.read_csv(INPUT_FILE, dtype=str, encoding="utf-8-sig", sep=delim)
    df.columns = [c.strip() for c in df.columns]

    if MATCH_COL not in df.columns:
        raise ValueError(f"Expected a '{MATCH_COL}' column in the input CSV.")

    columns = list(df.columns)
    output_rows = []

    n = len(df)
    i = 0
    while i < n:
        key_i = str(df.iloc[i][MATCH_COL]).strip()

        # Collect all consecutive rows sharing this MatchKey (could be
        # 2, 3, or more - not just pairs)
        j = i + 1
        while j < n and key_i != "" and str(df.iloc[j][MATCH_COL]).strip() == key_i:
            j += 1
        group = [df.iloc[k] for k in range(i, j)]

        if len(group) > 1:
            output_rows.append(pick_row(group, columns))
        else:
            row = group[0]
            single = {col: ("" if pd.isna(row[col]) else str(row[col]).strip()) for col in columns}
            single["Notes"] = "UNMATCHED: no adjacent row with same MatchKey - not combined"
            output_rows.append(single)

        i = j

    out_df = pd.DataFrame(output_rows, columns=columns + ["Notes"])
    # utf-8-sig writes a BOM so Excel (and other tools) correctly detect
    # UTF-8 instead of falling back to a local codepage like cp1252 -
    # that mismatch is what caused the garbled MatchKey text.
    out_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"Saved {OUTPUT_FILE} ({len(out_df)} row(s) from {n} input row(s))")


if __name__ == "__main__":
    main()


Saved /content/drive/MyDrive/Colab Notebooks/btaa_spc/SPC_overlap5_output.csv (113243 row(s) from 568475 input row(s))
